In [1]:
import pandas as pd

# Load the training and test datasets
train_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/07_sustainable_energy/train.csv'
test_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/DSEval/datasets/07_sustainable_energy/test.csv'

# Read the datasets
train_df = pd.read_csv(train_data_path)
test_df = pd.read_csv(test_data_path)

# Display the first few rows of the datasets
train_df.head(), test_df.head()


(                  Entity  ...   Longitude
 0  Saint Kitts and Nevis  ...  -62.782998
 1              Singapore  ...  103.819836
 2                  Qatar  ...   51.183884
 3             Mozambique  ...   35.529562
 4                Germany  ...   10.451526
 
 [5 rows x 20 columns],
        Entity  Access to electricity (% of population)  ...   Latitude  Longitude
 0      Guinea                                42.200000  ...   9.945587  -9.696645
 1       Yemen                                52.410000  ...  15.552727  48.516388
 2    Zimbabwe                                45.572647  ... -19.015438  29.154857
 3  Tajikistan                                98.633100  ...  38.861034  71.276093
 4       Spain                               100.000000  ...  40.463667  -3.749220
 
 [5 rows x 20 columns])

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Using the train_df DataFrame from the finished tasks
column_info_train = get_column_info(train_df)
print("Column information for train_df:")
print(column_info_train)

# Using the test_df DataFrame from the finished tasks
column_info_test = get_column_info(test_df)
print("Column information for test_df:")
print(column_info_test)


2025-08-30 19:01:41.124 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


Column information for train_df:
{'Category': ['Entity', 'Density\\n(P/Km2)'], 'Numeric': ['Access to electricity (% of population)', 'Access to clean fuels for cooking', 'Renewable-electricity-generating-capacity-per-capita', 'Financial flows to developing countries (US $)', 'Renewable energy share in the total final energy consumption (%)', 'Electricity from fossil fuels (TWh)', 'Electricity from nuclear (TWh)', 'Electricity from renewables (TWh)', 'Low-carbon electricity (% electricity)', 'Primary energy consumption per capita (kWh/person)', 'Energy intensity level of primary energy (MJ/$2017 PPP GDP)', 'Value_co2_emissions_kt_by_country', 'Renewables (% equivalent primary energy)', 'gdp_growth', 'gdp_per_capita', 'Land Area(Km2)', 'Latitude', 'Longitude'], 'Datetime': [], 'Others': []}
Column information for test_df:
{'Category': ['Entity', 'Density\\n(P/Km2)'], 'Numeric': ['Access to electricity (% of population)', 'Access to clean fuels for cooking', 'Renewable-electricity-genera

In [3]:
from metagpt.tools.libs.data_preprocess import LabelEncode
from metagpt.tools.libs.feature_engineering import GeneralSelection

# Copy the dataframes to avoid modifying the original data
train_df_copy = train_df.copy()
test_df_copy = test_df.copy()

# Define the target variable
target_variable = 'Renewable energy share in the total final energy consumption (%)'

# Initialize the GeneralSelection tool to drop unnecessary columns
selector = GeneralSelection(label_col=target_variable)

# Fit and transform the training data
train_df_processed = selector.fit_transform(train_df_copy)

# Transform the test data
test_df_processed = selector.transform(test_df_copy)

# Initialize the LabelEncode tool for categorical columns
label_encoder = LabelEncode(features=['Entity', 'Density\\n(P/Km2)'])

# Fit and transform the training data
train_df_processed = label_encoder.fit_transform(train_df_processed)

# Transform the test data
test_df_processed = label_encoder.transform(test_df_processed)

# Display the processed data
train_df_processed.head(), test_df_processed.head()


(   Entity  ...  Renewable energy share in the total final energy consumption (%)
 0     133  ...                                               1.60               
 1     143  ...                                               0.84               
 2     130  ...                                               0.05               
 3     107  ...                                              92.73               
 4      62  ...                                               4.41               
 
 [5 rows x 20 columns],
    Entity  ...  Renewable energy share in the total final energy consumption (%)
 0      67  ...                                              65.44               
 1     173  ...                                               1.08               
 2     175  ...                                              80.23               
 3     156  ...                                              64.07               
 4     150  ...                                              14.79      

In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

# Using the latest processed DataFrames
column_info_train = get_column_info(train_df_processed)
column_info_test = get_column_info(test_df_processed)

print("Column Info for Train DataFrame")
print(column_info_train)
print("\nColumn Info for Test DataFrame")
print(column_info_test)


Column Info for Train DataFrame
{'Category': [], 'Numeric': ['Entity', 'Access to electricity (% of population)', 'Access to clean fuels for cooking', 'Renewable-electricity-generating-capacity-per-capita', 'Financial flows to developing countries (US $)', 'Electricity from fossil fuels (TWh)', 'Electricity from nuclear (TWh)', 'Electricity from renewables (TWh)', 'Low-carbon electricity (% electricity)', 'Primary energy consumption per capita (kWh/person)', 'Energy intensity level of primary energy (MJ/$2017 PPP GDP)', 'Value_co2_emissions_kt_by_country', 'Renewables (% equivalent primary energy)', 'gdp_growth', 'gdp_per_capita', 'Density\\n(P/Km2)', 'Land Area(Km2)', 'Latitude', 'Longitude', 'Renewable energy share in the total final energy consumption (%)'], 'Datetime': [], 'Others': []}

Column Info for Test DataFrame
{'Category': [], 'Numeric': ['Entity', 'Access to electricity (% of population)', 'Access to clean fuels for cooking', 'Renewable-electricity-generating-capacity-per-

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# Define the target variable and features
X_train = train_df_processed.drop(columns=['Entity', 'Year', target_variable])
y_train = train_df_processed[target_variable]
X_test = test_df_processed.drop(columns=['Entity', 'Year', target_variable])
y_test = test_df_processed[target_variable]

# Initialize and train the linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model using RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
rmse


KeyError: "['Year'] not found in axis"